In [11]:
import os
import keras
from keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
seed = 0
np.random.seed(seed)
import tensorflow as tf
tf.random.set_seed(seed)

from keras.models import Sequential
from keras.optimizers import Adam
from keras.losses import SparseCategoricalCrossentropy
from hgq.config import QuantizerConfig, QuantizerConfigScope
from hgq.layers import QDense, QSoftmax
from hgq.utils.sugar import FreeEBOPs, PBar

In [12]:
import scipy.io
import numpy as np

def load_svhn(path):
    data = scipy.io.loadmat(path)
    X = data["X"]
    y = data["y"].reshape(-1)

    X = np.transpose(X, (3,0,1,2))

    y[y == 10] = 0

    return X, y

In [13]:
import gc
from hgq.utils.sugar import Dataset

X_train, y_train = load_svhn("/home/slopin/DAT255-project/SVHN/Data/train_32x32.mat")
X_test, y_test   = load_svhn("/home/slopin/DAT255-project/SVHN/Data/test_32x32.mat")

#Adding the extra data, but only a subset of it to avoid memory issues.
X_ex_tmp, y_ex_tmp = load_svhn("/home/slopin/DAT255-project/SVHN/Data/extra_32x32.mat")
indices = np.random.choice(len(X_ex_tmp), 100000, replace=False)
X_extra_sampled = X_ex_tmp[indices]
y_extra_sampled = y_ex_tmp[indices]

#Garbage collecting to free up memory.
del X_ex_tmp, y_ex_tmp
gc.collect()

#Concatenating the extra data with the training data.
X_train = np.concatenate((X_train, X_extra_sampled), axis=0)
y_train = np.concatenate((y_train, y_extra_sampled), axis=0)

X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

y_train = keras.utils.to_categorical(y_train, 10)
y_test  = keras.utils.to_categorical(y_test, 10)

dataset_train = Dataset(X_train, y_train, batch_size=2048, device='gpu:0', shuffle=True)
dataset_test = Dataset(X_test, y_test, batch_size=2048, device='gpu:0')
input_shape = (32, 32, 3)

2026-04-29 11:02:38.457597: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2128982016 exceeds 10% of free system memory.
2026-04-29 11:02:44.934492: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 319881216 exceeds 10% of free system memory.


In [15]:
import keras
from hgq.layers import QDense, QConv2D, QBatchNormDense, QEinsumDenseBatchnorm
from hgq.config import LayerConfigScope, QuantizerConfigScope


with (
      QuantizerConfigScope(place='all', default_q_type='kbi', f0=6, i0=4, overflow_mode='SAT_SYM', trainable=True),
      LayerConfigScope(enable_ebops=True, beta0=1e-6),
   ):
    
    inputs = keras.Input(shape=input_shape)

    # Conv Block (Homogeneous across axis (0,1,2,3) for activations) 
    with QuantizerConfigScope(place='datalane', default_q_type='kif', i0 = 1, overflow_mode='WRAP', heterogeneous_axis=None, homogeneous_axis=(0, 1, 2, 3)):
        x = QConv2D(32, (3, 3), activation='relu', kernel_regularizer=keras.regularizers.L2(1e-3))(inputs)
        x = keras.layers.Activation('relu')(x)
        x = keras.layers.MaxPooling2D((2, 2))(x)
        
        x = QConv2D(64, (3, 3), activation='relu', kernel_regularizer=keras.regularizers.L2(1e-3))(x) # Note: linked to previous x, not inputs
        x = keras.layers.Activation('relu')(x)
        x = keras.layers.MaxPooling2D((2, 2))(x)

        x = QConv2D(128, (3, 3), activation='relu', kernel_regularizer=keras.regularizers.L2(1e-3))(x) # Note: linked to previous x, not inputs
        x = keras.layers.MaxPooling2D((2, 2))(x)

    # Dense Block (Homogeneous across (0,1) for activations) 
    x = keras.layers.Flatten()(x)
    
    with QuantizerConfigScope(place='datalane', default_q_type='kif', i0 = 1, overflow_mode='WRAP', heterogeneous_axis=None, homogeneous_axis=(0, 1)):
        x = QEinsumDenseBatchnorm('bc,cC->bC',256, bias_axes='C', activation='relu', kernel_regularizer=keras.regularizers.L2(1e-3))(x)
        x = keras.layers.Dropout(0.2)(x)
        outputs = QDense(10)(x)

    model = keras.Model(inputs=inputs, outputs=outputs)

In [16]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_conv2d_7 (QConv2D)            │ (None, 30, 30, 32)     │         3,590 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_conv2d_8 (QConv2D)            │ (None, 13, 13, 64)     │        73,990 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 13, 13, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_conv2d_9 (QConv2D)            │ (None, 4, 4, 128)      │       295,430 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 2, 2, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_einsum_dense_batchnorm_1      │ (None, 256)            │       526,086 │
│ (QEinsumDenseBatchnorm)         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense_1 (QDense)              │ (None, 10)             │        10,286 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 909,382 (2.82 MB)

 Trainable params: 681,699 (2.60 MB)

 Non-trainable params: 227,683 (223.91 KB)

In [21]:
import keras
import numpy as np
from math import cos, pi
from hgq.utils.sugar import BetaScheduler, Dataset, FreeEBOPs, ParetoFront, PBar, PieceWiseSchedule, EarlyStoppingWithEbopsThres
from keras.callbacks import LearningRateScheduler, CSVLogger

OUTPUT_PATH = 'model_outputs'

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)
    print(f"Created new folder: {OUTPUT_PATH}")

def cosine_decay_restarts_schedule(
    initial_lr, first_decay_steps, m_mul, alpha
):
    def schedule(step):
        cycle = step // first_decay_steps
        x = (step % first_decay_steps) / first_decay_steps
        lr = alpha + 0.5 * (initial_lr * (m_mul ** cycle) - alpha) * (1 + cos(pi * x))
        return lr
    return schedule
def warmup_cosine_schedule(
    initial_lr,
    warmup_steps,
    first_decay_steps,
    m_mul=0.9,
    alpha=1e-5
):
    cosine = cosine_decay_restarts_schedule(
        initial_lr,
        first_decay_steps,
        m_mul=m_mul,
        alpha=alpha
    )

    def schedule(step):
        # Warmup phase
        if step < warmup_steps:
            return initial_lr * (0.1 + 0.9 * step / warmup_steps)

        # Cosine phase
        return cosine(step - warmup_steps)

    return schedule

pbar = PBar(
        'loss: {loss:.2f}/{val_loss:.2f} - acc: {accuracy:.4f}/{val_accuracy:.4f}' #lr: {learning_rate:.2e} - beta: {beta:.1e}
    )
ebops = FreeEBOPs()
pareto = ParetoFront(
        OUTPUT_PATH,
        ['val_accuracy', 'ebops'],
        [1, -1],
        fname_format='epoch={epoch}-val_acc={val_accuracy:.3f}-ebops={ebops}-val_loss={val_loss:.3f}.keras',
    )
estop = EarlyStoppingWithEbopsThres(ebops_threshold=3000, monitor='val_accuracy', patience=30, restore_best_weights=True)
beta_sched = BetaScheduler(
    PieceWiseSchedule([
        (0,    1e-7, 'log'),
        (50,   1e-6, 'log'),
        (500,  1e-5, 'log'),
        (5000, 1e-4, 'constant')
    ])
)
lr_sched = LearningRateScheduler(
    warmup_cosine_schedule(
        initial_lr=1e-3,
        warmup_steps=10,
        first_decay_steps=50,
        m_mul=0.9,
        alpha=1e-5
    )
)

csv_logger = CSVLogger('SVHN_HGQ_log.csv', append=True, separator=';')

In [22]:
callbacks = [ebops, pbar, pareto, estop, csv_logger]
#lr_sched, beta_sched,

In [25]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate = 1e-3),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
    jit_compile=True,
    steps_per_execution=10
)

In [ ]:
model.fit(dataset_train, epochs=5000, validation_data=dataset_test, callbacks=callbacks, verbose=0)

  0%|          | 1/5000 [13:08<1094:20:19, 788.08s/epoch]


In [ ]:
#pull model, search for best model can be automated, but is done manually in this case.
#Aiming for the lowest EBOPs with the highest accuracy.
from keras.models import load_model

model = load_model('/home/slopin/DAT255-project/Modeller/model_HGQ/model_outputs_StaticTraining_CNN/epoch=4468-val_acc=0.965-ebops=34215-val_loss=0.128.keras')

score = model.evaluate(dataset_test)
model.save('/home/slopin/DAT255-project/Modeller/model_HGQ/MNIST_CNN_HGQ_StaticTraining_model.keras')
print("Test loss:", score[0])
print("Test accuracy:", score[1])